# SVM Ensemble: Bagging (parallel) and Boosting (sequential) on Iris
Bagging and Boosting are written from scratch. SVM (`SVC`) is the only base classifier.

In [ ]:
import csv
import time

import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from matplotlib.colors import ListedColormap
from sklearn.svm import SVC
from sklearn.metrics.pairwise import rbf_kernel     # only to check rbf() below

## Data
Iris, 150 samples, 3 classes. Stratified 80 / 20 split (120 train / 30 test). Note: `sepal-length` and `sepal-width` in this file are rounded to integers.

In [ ]:
with open('iris_train.csv', encoding='utf-8') as f:
  rows = list(csv.reader(f))[1:]                          # skip header
X_all = np.array([[float(v) for v in r[:4]] for r in rows])   # 150 x 4
classes = sorted(set(r[4] for r in rows))                 # 3 class names
y_all = np.array([classes.index(r[4]) for r in rows])     # 150, values 0 / 1 / 2


def split(seed):
  # stratified 80 / 20: per class 40 train, 10 test
  rng = np.random.RandomState(seed)
  tr, te = [], []
  for c in range(3):
    idx = rng.permutation(np.where(y_all == c)[0])        # 50 indexes of class c
    tr += list(idx[:40])
    te += list(idx[40:])
  tr, te = np.array(tr), np.array(te)
  mu, sd = X_all[tr].mean(axis=0), X_all[tr].std(axis=0)  # scale from train only
  return (X_all[tr] - mu) / sd, y_all[tr], (X_all[te] - mu) / sd, y_all[te]


Xtr, ytr, Xte, yte = split(0)                             # main split (seed 0)
print('samples =', len(X_all), '| classes =', classes)
print('train =', Xtr.shape, '| test =', Xte.shape)
print('sepal-length values:', np.unique(X_all[:, 0]), '| sepal-width values:', np.unique(X_all[:, 1]))

## SVM
Soft margin: $\min \tfrac12\|w\|^2 + C\sum_i \xi_i$ s.t. $y_i(w^T x_i + b) \ge 1 - \xi_i,\ \xi_i \ge 0$

RBF kernel: $K(x,z) = \exp(-\gamma\|x-z\|^2)$, with $\|x-z\|^2 = \|x\|^2 + \|z\|^2 - 2x^Tz$

In [ ]:
def make_svm():
  return SVC(kernel='rbf', C=1.0, gamma=0.25)             # same SVM everywhere


def rbf(A, B, gamma):                                     # RBF kernel
  AA = np.sum(A ** 2, axis=1)[:, None]                    # n x 1
  BB = np.sum(B ** 2, axis=1)[None, :]                    # 1 x m
  return np.exp(-gamma * (AA + BB - 2 * A @ B.T))         # n x m


print('max |rbf - library| =', np.abs(rbf(Xtr[:10], Xtr[:10], 0.25) - rbf_kernel(Xtr[:10], Xtr[:10], gamma=0.25)).max())
svm = make_svm().fit(Xtr, ytr)
print('single SVM: support vectors =', len(svm.support_),
      '| test accuracy =', round(np.mean(svm.predict(Xte) == yte) * 100, 1), '%')

## Bagging (parallel)
Bootstrap $T$ sets (with replacement), train one SVM per set in parallel, majority vote: $\hat y(x) = \arg\max_c \sum_t \mathbb{1}[h_t(x) = c]$

In [ ]:
def make_boots(N, T, seed):
  rng = np.random.RandomState(seed)
  return [rng.randint(0, N, N) for _ in range(T)]        # T bootstrap index sets, N each


def fit_boot(Xb, yb):
  return make_svm().fit(Xb, yb)                          # train 1 SVM (runs in a worker)


def bagging_serial(X, y, T, seed):
  return [fit_boot(X[b], y[b]) for b in make_boots(len(X), T, seed)]


def bagging_parallel(X, y, T, seed, n_jobs):
  boots = make_boots(len(X), T, seed)                    # sampled in the main process
  return Parallel(n_jobs=n_jobs)(delayed(fit_boot)(X[b], y[b]) for b in boots)


def bagging_predict(models, X, upto=None):
  P = np.array([m.predict(X) for m in models[:upto]])    # T x n, vote of each model
  return np.array([np.bincount(P[:, j], minlength=3).argmax() for j in range(P.shape[1])])


T = 30
ms = bagging_serial(Xtr, ytr, T, 0)
mp = bagging_parallel(Xtr, ytr, T, 0, 4)
same = np.array_equal(bagging_predict(ms, Xte), bagging_predict(mp, Xte)) and \
       np.array_equal(bagging_predict(ms, Xtr), bagging_predict(mp, Xtr))
print('serial == parallel :', same)
print('Bagging T =', T, '| test accuracy =', round(np.mean(bagging_predict(mp, Xte) == yte) * 100, 1), '%')

## Boosting (sequential, AdaBoost SAMME)
Start with $w_i = 1/N$. Round $t$: train SVM $h_t$ with weights $w$, $\text{err}_t = \sum_i w_i \mathbb{1}[h_t(x_i) \ne y_i]$, $\alpha_t = \ln\frac{1-\text{err}_t}{\text{err}_t} + \ln(K-1)$, $w_i \leftarrow w_i e^{\alpha_t \mathbb{1}[h_t(x_i) \ne y_i]}$ (then normalize). Predict: $\hat y(x) = \arg\max_c \sum_t \alpha_t \mathbb{1}[h_t(x) = c]$

Weights go into `sample_weight = N * w`, which scales $C$ per sample: $\min \tfrac12\|w\|^2 + C\sum_i N w_i \xi_i$. Stop if $\text{err}_t = 0$, or if $\text{err}_t \ge 1 - 1/K$ (that model is dropped).

In [ ]:
def boosting_fit(X, y, T, K=3):
  N = len(X)
  w = np.ones(N) / N                                      # equal weights
  models, alphas, errs, ws, wrongs = [], [], [], [], []
  for t in range(T):
    m = make_svm().fit(X, y, sample_weight=N * w)         # large weight = large C for that sample
    wrong = m.predict(X) != y                             # N, True if misclassified
    err = np.sum(w * wrong)                               # weighted error
    errs.append(err)
    if err >= 1 - 1 / K:                                  # worse than guessing: drop and stop
      break
    e = max(err, 1e-10)                                   # avoid log(0)
    alpha = np.log((1 - e) / e) + np.log(K - 1)
    models.append(m)
    alphas.append(alpha)
    ws.append(w.copy())
    wrongs.append(wrong)
    if err == 0:                                          # perfect on train: stop
      break
    w = w * np.exp(alpha * wrong)                         # raise weight of wrong samples
    w = w / w.sum()                                       # sum = 1
  return models, alphas, errs, ws, wrongs


def boosting_predict(models, alphas, X, upto=None):
  S = np.zeros((len(X), 3))                               # n x 3, score of each class
  for m, a in list(zip(models, alphas))[:upto]:
    S[np.arange(len(X)), m.predict(X)] += a               # weighted vote
  return S.argmax(axis=1)


T = 30
mb, ab, eb, wsb, wrb = boosting_fit(Xtr, ytr, T)
print('Boosting rounds used =', len(mb), 'of', T)
print('Boosting T =', T, '| test accuracy =', round(np.mean(boosting_predict(mb, ab, Xte) == yte) * 100, 1), '%')
print('round 1: wrong =', int(wrb[0].sum()), 'of', len(Xtr),
      '| mean weight of wrong samples:', round(wsb[0][wrb[0]].mean(), 4), '->', round(wsb[1][wrb[0]].mean(), 4))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].bar(range(1, len(eb) + 1), eb, color='tab:red')
ax[0].set_title('Weighted error per round')
ax[0].set_xlabel('round'); ax[0].set_ylabel('err')
ax[1].bar(range(1, len(ab) + 1), ab, color='tab:green')
ax[1].set_title('Model weight alpha per round')
ax[1].set_xlabel('round'); ax[1].set_ylabel('alpha')
plt.tight_layout()
plt.show()

## Comparison (20 random 80 / 20 splits, T = 30)

In [ ]:
N_SEEDS, T, N_JOBS = 20, 30, 4
TS = [1, 2, 3, 5, 10, 15, 20, 30]                         # number of models used

acc_svm = np.zeros(N_SEEDS)
acc_bag = np.zeros((N_SEEDS, len(TS)))
acc_boo = np.zeros((N_SEEDS, len(TS)))
for s in range(N_SEEDS):
  Xa, ya, Xb, yb = split(s)
  acc_svm[s] = np.mean(make_svm().fit(Xa, ya).predict(Xb) == yb)
  models = bagging_parallel(Xa, ya, T, s, N_JOBS)         # Bagging (parallel)
  mb_, ab_, _, _, _ = boosting_fit(Xa, ya, T)             # Boosting (sequential)
  for j, n in enumerate(TS):
    acc_bag[s, j] = np.mean(bagging_predict(models, Xb, n) == yb)
    acc_boo[s, j] = np.mean(boosting_predict(mb_, ab_, Xb, n) == yb)
  if s == 0:                                              # keep seed 0 predictions for confusion matrix
    pred_bag0 = bagging_predict(models, Xb)
    pred_boo0 = boosting_predict(mb_, ab_, Xb)
    yte0 = yb

last = len(TS) - 1                                        # column of T = 30
print(f'{"method":26s} {"seed 0":>8s} {"mean":>8s} {"SD":>6s}')
for name, a0, a in [('SVM (single)', acc_svm[0], acc_svm),
                    (f'Bagging T={T} (parallel)', acc_bag[0, last], acc_bag[:, last]),
                    (f'Boosting T={T}', acc_boo[0, last], acc_boo[:, last])]:
  print(f'{name:26s} {a0 * 100:7.1f}% {np.mean(a) * 100:7.1f}% {np.std(a) * 100:5.1f}%')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

means = [acc_svm.mean(), acc_bag[:, last].mean(), acc_boo[:, last].mean()]
sds = [acc_svm.std(), acc_bag[:, last].std(), acc_boo[:, last].std()]
ax[0].bar(['SVM', 'Bagging', 'Boosting'], np.array(means) * 100, yerr=np.array(sds) * 100,
          capsize=6, color=['gray', 'tab:blue', 'tab:orange'])
ax[0].set_ylabel('Accuracy (%)')
ax[0].set_title('Mean accuracy over 20 seeds (error bar = SD)')

for acc, name, color in [(acc_bag, 'Bagging', 'tab:blue'), (acc_boo, 'Boosting', 'tab:orange')]:
  m, sd = acc.mean(axis=0) * 100, acc.std(axis=0) * 100
  ax[1].plot(TS, m, 'o-', color=color, label=name)
  ax[1].fill_between(TS, m - sd, m + sd, color=color, alpha=0.15)
ax[1].axhline(acc_svm.mean() * 100, color='gray', linestyle='--', label='SVM (single)')
ax[1].set_xlabel('number of models T')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_title('Accuracy vs number of models (band = SD)')
ax[1].legend()
plt.tight_layout()
plt.show()


def confusion(y_true, y_pred):                            # rows = true, columns = predicted
  cm = np.zeros((3, 3), dtype=int)
  for t, p in zip(y_true, y_pred):
    cm[t, p] += 1
  return cm


fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
for a, pred, name in [(ax[0], pred_bag0, 'Bagging'), (ax[1], pred_boo0, 'Boosting')]:
  cm = confusion(yte0, pred)
  a.imshow(cm, cmap='Blues')
  for r in range(3):
    for c in range(3):
      a.text(c, r, cm[r, c], ha='center', va='center', color='white' if cm[r, c] > 5 else 'black')
  a.set_xticks(range(3)); a.set_yticks(range(3))
  a.set_xticklabels(['setosa', 'versicolor', 'virginica'], rotation=20)
  a.set_yticklabels(['setosa', 'versicolor', 'virginica'])
  a.set_xlabel('predicted'); a.set_ylabel('true')
  a.set_title(f'{name} confusion matrix (seed 0)')
plt.tight_layout()
plt.show()

## Training time: Bagging serial vs parallel
Workers are warmed up before timing.

In [ ]:
Xa, ya, _, _ = split(0)
Ts = [50, 500, 2000, 5000]                                # number of models
workers = [2, 4, 8]

t_serial = []
for T_ in Ts:
  t0 = time.time()
  bagging_serial(Xa, ya, T_, 0)
  t_serial.append(time.time() - t0)

t_par = {n: [] for n in workers}
for n in workers:
  bagging_parallel(Xa, ya, 2 * n, 0, n)                   # warm up: start workers first
  for T_ in Ts:
    t0 = time.time()
    bagging_parallel(Xa, ya, T_, 0, n)
    t_par[n].append(time.time() - t0)

print(f'{"T":>6s} {"serial":>9s}' + ''.join(f'{str(n) + " workers":>12s}' for n in workers))
for i, T_ in enumerate(Ts):
  print(f'{T_:6d} {t_serial[i]:8.2f}s' + ''.join(f'{t_par[n][i]:11.2f}s' for n in workers))
print('speedup (serial / parallel):')
for i, T_ in enumerate(Ts):
  print(f'T = {T_:5d} :', '  '.join(f'{n} workers = {t_serial[i] / t_par[n][i]:.2f}x' for n in workers))

fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(len(Ts))
ax.bar(x - 0.3, t_serial, 0.2, label='serial', color='gray')
for k, n in enumerate(workers):
  ax.bar(x - 0.1 + 0.2 * k, t_par[n], 0.2, label=f'parallel {n} workers')
ax.set_xticks(x)
ax.set_xticklabels([f'T={t}' for t in Ts])
ax.set_yscale('log')
ax.set_ylabel('time (seconds, log scale)')
ax.set_title('Bagging training time: serial vs parallel')
ax.legend()
plt.tight_layout()
plt.show()

## Inference time: predicting the test set
Every model in an ensemble must predict, so prediction time grows with the number of models $T$. The 30 test rows are repeated 100 times (3000 rows) to get a stable timing.

In [ ]:
Xa, ya, Xb, yb = split(0)
Xbig = np.tile(Xb, (100, 1))                              # 3000 x 4, test set repeated 100 times

svm_ = make_svm().fit(Xa, ya)
bag_ = bagging_serial(Xa, ya, 30, 0)                      # 30 SVMs
boo_, boo_a, _, _, _ = boosting_fit(Xa, ya, 30)           # 30 rounds


def avg_time(f, n=10):                                    # mean seconds of n runs
  f()                                                     # warm up
  t0 = time.perf_counter()
  for _ in range(n):
    f()
  return (time.perf_counter() - t0) / n


t_svm = avg_time(lambda: svm_.predict(Xbig))
t_bag = avg_time(lambda: bagging_predict(bag_, Xbig))
t_boo = avg_time(lambda: boosting_predict(boo_, boo_a, Xbig))
print(f'{"method":16s} {"time (ms)":>10s} {"vs single":>10s} {"support vectors / model":>25s}')
for name, t, sv in [('SVM (single)', t_svm, [len(svm_.support_)]),
                    ('Bagging T=30', t_bag, [len(m.support_) for m in bag_]),
                    ('Boosting T=30', t_boo, [len(m.support_) for m in boo_])]:
  print(f'{name:16s} {t * 1000:10.1f} {t / t_svm:9.1f}x {np.mean(sv):25.1f}')

Ts_inf = [1, 5, 10, 20, 30]                               # number of models used
t_bag_T = [avg_time(lambda: bagging_predict(bag_, Xbig, n)) for n in Ts_inf]
t_boo_T = [avg_time(lambda: boosting_predict(boo_, boo_a, Xbig, n)) for n in Ts_inf]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(['SVM', 'Bagging', 'Boosting'], np.array([t_svm, t_bag, t_boo]) * 1000,
          color=['gray', 'tab:blue', 'tab:orange'])
ax[0].set_ylabel('time (ms) for 3000 rows')
ax[0].set_title('Inference time (T = 30)')

ax[1].plot(Ts_inf, np.array(t_bag_T) * 1000, 'o-', color='tab:blue', label='Bagging')
ax[1].plot(Ts_inf, np.array(t_boo_T) * 1000, 'o-', color='tab:orange', label='Boosting')
ax[1].axhline(t_svm * 1000, color='gray', linestyle='--', label='SVM (single)')
ax[1].set_xlabel('number of models T')
ax[1].set_ylabel('time (ms) for 3000 rows')
ax[1].set_title('Inference time vs number of models')
ax[1].legend()
plt.tight_layout()
plt.show()

## Decision boundary (petal-length, petal-width)
Black circles = support vectors of the single SVM.

In [ ]:
Xa, ya, _, _ = split(0)
X2 = Xa[:, 2:4]                                           # petal-length, petal-width: 120 x 2

svm2 = make_svm().fit(X2, ya)
bag2 = bagging_serial(X2, ya, 30, 0)
boo2, boo2_a, _, _, _ = boosting_fit(X2, ya, 30)

gx, gy = np.meshgrid(np.linspace(X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5, 150),
                     np.linspace(X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5, 150))
G = np.c_[gx.ravel(), gy.ravel()]                         # 22500 x 2, every grid point

cmap_bg = ListedColormap(['#bbdefb', '#ffe0b2', '#c8e6c9'])
cmap_pt = ListedColormap(['#1565c0', '#ef6c00', '#2e7d32'])
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for a, pred, name in [(ax[0], svm2.predict(G), 'SVM (single)'),
                      (ax[1], bagging_predict(bag2, G), 'Bagging (30 SVMs)'),
                      (ax[2], boosting_predict(boo2, boo2_a, G), 'Boosting (SVM, SAMME)')]:
  a.contourf(gx, gy, pred.reshape(gx.shape), cmap=cmap_bg, alpha=0.9)
  a.scatter(X2[:, 0], X2[:, 1], c=ya, cmap=cmap_pt, edgecolor='white', s=28)
  a.set_xlabel('petal-length (standardized)')
  a.set_ylabel('petal-width (standardized)')
  a.set_title(name)
ax[0].scatter(svm2.support_vectors_[:, 0], svm2.support_vectors_[:, 1],
              s=90, facecolors='none', edgecolors='black', linewidths=1.2)
plt.tight_layout()
plt.show()